# Nestlé Commodity Price Pipeline — Alpha Vantage API

**Goal:** Pull commodity price data via the Alpha Vantage API as an alternative
to the World Bank Excel file. Clean, convert units, and visualise price trends.

**Commodities covered:** Coffee Arabica, Sugar (World), Wheat, Corn (Maize)

**Note on data source choice:**
Alpha Vantage is better suited for near-real-time updates (free tier: 25 calls/day).
For deep historical analysis (pre-1990), the World Bank Pink Sheet remains more reliable.
The ideal architecture: World Bank for historical baseline, Alpha Vantage to top up recent months.

**Pipeline stages:**
1. Fetch data from Alpha Vantage API
2. Parse JSON responses into DataFrames
3. Convert units to a common standard ($/kg or $/mt)
4. Combine into a single clean DataFrame
5. Visualise indexed price trends


---
## 1. Fetch Data from Alpha Vantage API

API key is loaded from an environment variable — never hardcode credentials in a notebook.
Set it once in your terminal before launching Jupyter:
```bash
export AV_API_KEY="your_key_here"
```

**Rate limit note:** The free tier allows 25 requests/day and ~5 requests/minute.
A `time.sleep(15)` between calls prevents hitting the per-minute limit.


In [4]:
import requests
import pandas as pd
import time

# Load API key from environment — never hardcode credentials in a notebook
API_KEY = "OH892Q89PFJ65S1K"

def fetch_commodity(function: str, api_key: str):
    """Fetch monthly commodity prices from Alpha Vantage.

    Returns a DataFrame with columns: date, value, commodity_function
    Returns None if the API returns an error.
    """
    response = requests.get(
        "https://www.alphavantage.co/query",
        params={"function": function, "interval": "monthly", "apikey": api_key}
    )
    data = response.json()

    if "data" not in data:
        print(f"  ERROR fetching {function}: {data}")
        return None

    print(f"{function}: {len(data['data'])} rows | unit = {data.get('unit', 'unknown')}")

    df = pd.DataFrame(data["data"])
    df["date"]               = pd.to_datetime(df["date"])
    # Alpha Vantage uses "." as a missing value placeholder
    df["value"]              = pd.to_numeric(df["value"], errors="coerce")
    df["commodity_function"] = function
    return df


# Fetch all four commodities with a sleep between calls to respect rate limits
df_coffee_raw = fetch_commodity("COFFEE", API_KEY)
time.sleep(15)
df_sugar_raw  = fetch_commodity("SUGAR",  API_KEY)
time.sleep(15)
df_wheat_raw  = fetch_commodity("WHEAT",  API_KEY)
time.sleep(15)
df_corn_raw   = fetch_commodity("CORN",   API_KEY)


COFFEE: 553 rows | unit = cents per pound
SUGAR: 553 rows | unit = cents per pound
WHEAT: 553 rows | unit = dollar per metric ton
CORN: 553 rows | unit = dollar per metric ton


---
## 2. Unit Conversion

Alpha Vantage returns all four commodities in **cents/pound (cents/lb)**.
We convert everything to a common standard matching the World Bank Pink Sheet:
- Coffee, Sugar → **$/kg** (`cents/lb ÷ 100 ÷ 0.453592`)
- Wheat, Corn → **$/mt** (`cents/lb ÷ 100 ÷ 0.453592 × 1000`)

Both formulas use the same base conversion (cents → dollars, pounds → kg),
then Wheat and Corn are scaled by ×1000 to go from $/kg to $/mt.


In [5]:
def cents_per_lb_to_usd_per_kg(series: pd.Series) -> pd.Series:
    """Convert cents/lb to $/kg."""
    return (series / 100 / 0.453592).round(4)

def cents_per_lb_to_usd_per_mt(series: pd.Series) -> pd.Series:
    """Convert cents/lb to $/mt."""
    return (series / 100 / 0.453592 * 1000).round(4)


# Apply conversions and add commodity labels
df_coffee_raw["price"]     = cents_per_lb_to_usd_per_kg(df_coffee_raw["value"])
df_coffee_raw["commodity"] = "Coffee Arabica"

df_sugar_raw["price"]      = cents_per_lb_to_usd_per_kg(df_sugar_raw["value"])
df_sugar_raw["commodity"]  = "Sugar (World)"

df_wheat_raw["price"]      = cents_per_lb_to_usd_per_mt(df_wheat_raw["value"])
df_wheat_raw["commodity"]  = "Wheat"

df_corn_raw["price"]       = cents_per_lb_to_usd_per_mt(df_corn_raw["value"])
df_corn_raw["commodity"]   = "Corn (Maize)"

# Sanity check on a known value — coffee spot price in 2024 was ~$4-5/kg
print("Coffee price sample (should be ~$/kg):")
print(df_coffee_raw[["date", "value", "price"]].dropna().tail(3))


Coffee price sample (should be ~$/kg):
          date      value   price
274 2003-03-01  61.748095  1.3613
275 2003-02-01  66.408500  1.4641
276 2003-01-01  65.383478  1.4415


---
## 3. Combine and Clean

Stack all four DataFrames into one long-format table,
drop nulls (Alpha Vantage backfills missing history with `.` placeholders),
and sort ascending so the indexing logic in the next step works correctly.

**Why sort matters for indexing:** the `index_to_100` function takes `iloc[0]`
as the base value. If the data is sorted descending (newest first, as the API returns it),
`iloc[0]` is the most recent price — giving a nonsensical baseline.


In [6]:
df_av = pd.concat([
    df_coffee_raw[["date", "commodity", "price"]],
    df_sugar_raw[ ["date", "commodity", "price"]],
    df_wheat_raw[ ["date", "commodity", "price"]],
    df_corn_raw[  ["date", "commodity", "price"]],
], ignore_index=True)

df_av["source"] = "Alpha Vantage"

# Drop nulls — removes placeholder rows from sparse historical coverage
df_av = df_av.dropna(subset=["price"])

# Sort ascending — required for correct index_to_100 baseline
df_av = df_av.sort_values(["commodity", "date"]).reset_index(drop=True)

# Verify shape and null count
print(df_av.groupby("commodity")["price"].agg(["count", "min", "max"]))
print(f"\nTotal nulls: {df_av['price'].isna().sum()}")
print(f"Date range : {df_av['date'].min()} -> {df_av['date'].max()}")


                count        min        max
commodity                                  
Coffee Arabica    277     1.3458     9.0319
Corn (Maize)      277  2059.4532  7683.2730
Sugar (World)     179     0.2216     0.6497
Wheat             277  2552.4249  9791.9842

Total nulls: 0
Date range : 2003-01-01 00:00:00 -> 2026-01-01 00:00:00


---
## 4. Visualise — Indexed Price Trends

Index all commodities to 100 at the start of the window so they can share
a single Y axis regardless of unit ($/kg coffee vs $/mt wheat).


In [7]:
import plotly.express as px

# Filter to last 26 years
cutoff     = df_av["date"].max() - pd.DateOffset(years=26)
df_filtered = df_av[df_av["date"] >= cutoff].copy()
df_filtered = df_filtered.sort_values(["commodity", "date"]).reset_index(drop=True)

# Index each commodity to 100 at its own earliest date in the window
frames = []
for c in df_filtered["commodity"].unique():
    group = df_filtered[df_filtered["commodity"] == c].copy().reset_index(drop=True)
    base  = group["price"].iloc[0]
    group["price_indexed"] = (group["price"] / base * 100).round(2)
    frames.append(group)

df_indexed = pd.concat(frames, ignore_index=True)

fig = px.line(
    df_indexed,
    x="date", y="price_indexed", color="commodity",
    title="Commodity Prices — Last 26 Years (Indexed to 100 at start)",
    labels={"price_indexed": "Price Index (base = 100)", "date": ""},
    template="plotly_white",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.add_hline(y=100, line_dash="dot", line_color="gray", opacity=0.4)
fig.update_layout(
    hovermode="x unified",
    legend=dict(title="", orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    font=dict(family="Arial", size=12),
    height=520,
)
fig.show()


---
## Notes & Limitations

- **Free tier cap:** 25 API calls/day, ~5/minute. The `time.sleep(15)` between calls
  prevents the per-minute limit but the daily cap means you cannot add many more commodities
  without upgrading or caching responses locally
- **Historical coverage:** Alpha Vantage commodity data goes back to ~1960 on paper
  but pre-1980 coverage is sparse — many rows return `.` (null after coerce)
- **Missing commodities:** Cocoa, Palm Oil, and Soybeans are not available
  on the Alpha Vantage free tier — use the World Bank Pink Sheet for those
- **Next step:** merge `df_av` with `df_long` from the World Bank pipeline
  on `(date, commodity)` to cross-validate prices between the two sources
